## Lattice data analysis

### Introduction

In this chapter, we will use the human breast cancer Xenium dataset [@Janesick2023-high-res], which we loaded above (using the `r BiocStyle::Biocpkg("STexampleData")` package [@Righelli2022-SpatialExperiment]) and converted into a `SpatialFeatureExperiment` object. The `r BiocStyle::Biocpkg("SpatialFeatureExperiment")` package integrates the `r BiocStyle::Biocpkg("SpatialExperiment")` class with geometric annotations that are represented as Simple Features in the `r BiocStyle::CRANpkg("sf")` packages. We will use the `r BiocStyle::Biocpkg("Voyager")` package [@Moses2023-Voyager], which provides convenient wrappers around classic geospatial R packages such as `r BiocStyle::CRANpkg("spdep")` [@Pebesma2023-book-spatial-data-science]. For more details, please consult the authors' comprehensive [vignettes](https://pachterlab.github.io/voyager/index.html). 

This part is based on the [*pasta* resource vignette on lattice data analysis](https://robinsonlabuzh.github.io/pasta/00-overview-latSOD.html) [@Emons2025-pasta].

### The spatial weight matrix

Lattice based spatial methods rely on the concept of neighborhood. The neighborhood defines the spatial dependency between locations. Therefore, the first step of lattice based spatial analysis is the construction of a spatial weight matrix. Here we will use a nearest neighbor-based approach.

::: {.callout-note collapse="true" title="Spatial weight matrix"}

Different methods for the construction of the weight matrix exist, such as

* contiguity-based neighbors (neighbors in direct contact),
* graph-based neighbors (e.g., k-nearest neighbors),
* distance-based neighbors,
* higher order neighbors.

A detailed overview can be found in the [documentation](https://r-spatial.github.io/spdep/articles/nb.html) of the `r BiocStyle::CRANpkg("spdep")` package.

As different weight matrices influence downstream results, analysts should justify their choice of the weight matrix. A more detailed overview can be found in @Pebesma2023-book-spatial-data-science.

:::

In [ ]:
# construct (K=6)NN-graph using cell centroids
knn6 <- findSpatialNeighbors(sfe, type="centroids", method="knearneigh", k=6)
colGraph(sfe, "knn6") <- knn6
# visualize across tissue section
plotColGraph(sfe, 
    colGraphName="knn6", 
    colGeometryName="centroids", 
    segment_size=0.1,
    geometry_size=0.1) + 
    theme_void()

### Spatial autocorrelation

Spatial autocorrelation measures similarity between spatial units (e.g., cells) while recognize that the units are not independent due to their spatial context. Spatial autocorrelation metrics can be global (summarizing the entire study area) of view or local (provide statistic for each unit). In addition, there exist methods for univariate and multivariate comparisons of continuous and categorical data [@Pebesma2023-book-spatial-data-science].

#### Univariate measures - Moran's $I$

Moran's $I$ can be interpreted as the Pearson correlation between the value at a certain location and the average values of its neighbors. The global value is a weighted average of the respective local values [@Moran1950-stochastic].

In [ ]:
# get gene probes & compute Moran's I for them
idx <- rowData(sfe)$Type == "Gene Expression"
length(geneProbes <- rowData(sfe)[idx, "Symbol"])
sfe <- runUnivariate(sfe, 
    type="moran", 
    features=geneProbes, 
    colGraphName="knn6", 
    BPPARAM=bp)

We can visualize the three genes with highest Moran's $I$.

In [ ]:
I <- rowData(sfe)$moran_sample01
o <- order(I, decreasing=TRUE)
topGenes <- rownames(sfe)[head(o, 3)]
plotSpatialFeature(sfe, topGenes, ncol=3)

We can further visualize this using when calculating and plotting local Moran's $I$ values [@Anselin1995-LISA]. The interpretation is analogue to the global counterpart. The higher the value, the more similar the expression among a cell's neighbors. Negative values indicate local dissimilarity in expression.
 

In [ ]:
sfe <- runUnivariate(sfe, 
    type="localmoran", 
    features=topGenes, 
    colGraphName="knn6", 
    BPPARAM=bp)
plotLocalResult(sfe,
    name="localmoran",
    features=topGenes,
    colGeometryName="centroids",
    divergent=TRUE,
    diverge_center=0,
    ncol=3)

::: {.callout-note collapse="true" title="Statistical interpretation of autocorrelation"}

During the interpretation of local autocorrelation measures, both the effect size (the value of the statistic) and the significance level should be considered. Because we calculate significance on each spot individually, values should be corrected for multiple testing.

In [ ]:
plotLocalResult(sfe,
    name="localmoran",
    features=topGenes,
    attribute="-log10p_adj",
    colGeometryName="centroids",
    divergent=TRUE,
    diverge_center=0,
    ncol=3)

:::

#### Multivariate measures – Lee's L

Lee's $L$ combines the Pearson correlation coefficient and Moran's $I$ [@Lee2001-bivariate]. This unified metric allows to evaluate how two continuous variables are related while accounting for their spatial dependencies. We will calculate the global metric on the 20 most highly (non-spatial) variable features.

In [ ]:
res <- calculateBivariate(sfe, type="lee", feature1=geneProbes, colGraphName="knn6")

We will identify the gene pairs with highest Lee's $L$ value 
and then calculate and visualize the respective local measure.

In [ ]:
# select the 20 highest values in the res matrix
val <- tail(sort(res), 20)[1]
genePairs <- which(res >= val, arr.ind=TRUE)
# keep only pairs containing different genes
genePairs <- genePairs[genePairs[,1] != genePairs[,2], ]
data.frame(
    val=res[genePairs],
    i=rownames(res)[genePairs[,1]], 
    j=colnames(res)[genePairs[,2]])

In [ ]:
sfe <- runBivariate(
    sfe,
    type="locallee",
    colGraphName= "knn6",
    feature1=c("EPCAM", "KRT7"))
plotLocalResult(
    sfe,
    name="locallee",
    features="KRT7__EPCAM",
    colGeometryName="centroids",
    divergent=TRUE, diverge_center=0)

### Join count statistics

Join count statistics can be used to quantify the arrangement of categorical marks on lattices. The join count statistic calculates how often a categorical mark appears next to itself or to another category within the pre-defined neighborhood. This values can then be compared against a theoretical or a permutation-based value test of the null hypothesis of random spatial allocation of the marks [@Getis2009-spatial-weights].

Here, we will use the this to quantify the tendency of clusters (non-spatial) to co-localize in their relative neighborhood. Note that the output contains z-scores and no p-values such that a user-defined threshold can be applied.

In [ ]:
# code adapted from 
# https://robinsonlabuzh.github.io/
# pasta/04-imaging-multivar-latSOD.html

# dependencies
library(BiocNeighbors)
library(BiocSingular)
library(bluster)
library(scater)

# log-library size normalization
sfe <- logNormCounts(sfe)

# set seed for random number generation
# in order to make results reproducible
set.seed(123)

# run PCA on the sample
sfe <- runPCA(sfe, exprs_values="logcounts", ncomponents=50)

# cluster based on first 10 PC's 
# using Leiden community detection
pcs <- reducedDim(sfe, "PCA")[, 1:10]
params <- KNNGraphParam(
    k=20,
    cluster.fun="leiden",
    cluster.args=list(
        resolution=0.3,
        objective_function="modularity"))
colData(sfe)$cluster <- clusterRows(pcs, BLUSPARAM=params)

# visualize cluster assignments
plotSpatialFeature(sfe, 
    features="cluster", colGeometryName="centroids") +
    guides(col=guide_legend(override.aes=list(size=2)))

In [ ]:
resJc <- joincount.multi(as.factor(sfe$cluster), colGraph(sfe, "knn6"))
resJc <- resJc[order(resJc[, "z-value"], decreasing=TRUE), ]
head(resJc, 20)

We note that the non-spatial cluster labels are of course most likely found next to each other. Apart from this obvious result, the first top non-self interaction is $5:3$. Looking at the plot of the spatial distributions of the clusters these are two clusters in the ductal carcinoma regions of the tissue, the carcinoma core and the border. 